## Tujuan
1. Memuat dataset besar secara efisien dengan **Polars lazy evaluation** (`scan_csv`).
2. Mengenali struktur data: dimensi, tipe kolom, rentang waktu, dan cakupan wilayah.
3. Menemukan masalah kualitas awal (missing values, duplikat, skala nilai, ketidakseimbangan kelas).

## Dataset
Flood Prediction Dataset, Sulawesi Selatan (Kaggle, `farahfirdausa/flood-prediction-dataset`).
Sumber, lisensi, dan deskripsi kolom lengkap ada di `data/README.md`.

## 1. Setup dan Lokasi Data
Sesuaikan `DATA_PATH` dengan lokasi file di folder `data/raw/`.

In [3]:
import os
import time
import polars as pl
import duckdb

DATA_PATH = "data/raw/modis_flood_features_paling cleaning.csv"

if not os.path.exists(DATA_PATH) and os.path.exists("../" + DATA_PATH):
    DATA_PATH = "../" + DATA_PATH

PARQUET_PATH = DATA_PATH.replace("data/raw", "data/processed").replace(".csv", ".parquet")

print("Polars :", pl.__version__)
print("DuckDB :", duckdb.__version__)
print("File ada:", os.path.exists(DATA_PATH))

Polars : 1.44.2
DuckDB : 1.5.6
File ada: True


## 2. Membaca Data Secara Lazy
`pl.scan_csv` untuk membuat **rencana eksekusi** (LazyFrame), belum membaca seluruh file ke memori.
Data baru dibaca saat `.collect()` dipanggil, sehingga Polars dapat mengoptimalkan query
(hanya membaca kolom dan baris yang diperlukan). 

In [4]:
lf = pl.scan_csv(DATA_PATH, try_parse_dates=True)
schema = lf.collect_schema()
print(f"Jumlah kolom: {len(schema)}")
schema

Jumlah kolom: 16


Schema([('date', Date),
        ('lon', Float64),
        ('lat', Float64),
        ('flooded', Float64),
        ('jrc_perm_water', Float64),
        ('precip_1d', Float64),
        ('precip_3d', Float64),
        ('NDVI', Float64),
        ('NDWI', Float64),
        ('landcover', Float64),
        ('elevation', Float64),
        ('slope', Float64),
        ('aspect', Float64),
        ('upstream_area', Float64),
        ('TWI', Float64),
        ('target', Int64)])

## 3. Dimensi dan Ukuran
Memastikan dataset memenuhi syarat tugas (minimal 500 MB atau lebih dari 1 juta baris).

In [5]:
n_rows = lf.select(pl.len()).collect().item()
size_mb = os.path.getsize(DATA_PATH) / 1024**2

print(f"Jumlah baris : {n_rows:,}")
print(f"Jumlah kolom : {len(schema)}")
print(f"Ukuran file  : {size_mb:.1f} MB")
print("Memenuhi syarat > 1 juta baris:", n_rows > 1_000_000)

Jumlah baris : 1,025,801
Jumlah kolom : 16
Ukuran file  : 179.4 MB
Memenuhi syarat > 1 juta baris: True


## 4. Sampel Data
Melihat beberapa baris awal untuk memahami isi tiap kolom.

In [6]:
lf.head(10).collect()

date,lon,lat,flooded,jrc_perm_water,precip_1d,precip_3d,NDVI,NDWI,landcover,elevation,slope,aspect,upstream_area,TWI,target
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
2003-12-10,120.478677,-6.495942,1.0,1.0,9.645125,28.935376,6190.93062,-0.051446,17.0,0.0,0.116685,270.0,0.03199,1.042098,0
2003-12-10,120.480923,-6.495942,1.0,1.0,9.645125,28.935376,6190.93062,-0.051446,17.0,1.0,0.839713,236.480074,0.03199,1.042098,0
2003-12-10,120.478677,-6.493697,0.0,1.0,9.645125,28.935376,6190.93062,0.028399,17.0,0.0,0.46673,270.0,0.03199,1.042098,0
2003-12-10,120.480923,-6.493697,1.0,0.0,9.645125,28.935376,6190.93062,0.028399,17.0,4.0,0.939179,240.413746,0.021235,0.258796,1
2003-12-10,120.478677,-6.491451,0.0,1.0,9.645125,28.935376,6190.93062,-0.002506,17.0,0.0,0.583403,270.0,0.03199,1.042098,0
2003-12-10,120.478677,-6.486959,1.0,1.0,9.645125,28.935376,6190.93062,0.025401,17.0,0.0,0.700067,270.0,0.03199,1.042098,0
2003-12-10,120.476431,-6.484713,1.0,1.0,9.645125,28.935376,6190.93062,0.025401,17.0,0.0,0.0,0.0,0.03199,1.042098,0
2003-12-10,120.478677,-6.484713,1.0,1.0,9.645125,28.935376,6190.93062,0.025401,17.0,0.0,0.816726,270.0,0.03199,1.042098,0
2003-12-10,120.480923,-6.484713,0.0,0.0,9.645125,28.935376,6190.93062,0.111974,11.0,7.0,1.166669,270.0,0.027606,0.304226,0


## 5. Missing Values
Menghitung jumlah nilai kosong (`null`) di setiap kolom.

In [7]:
lf.null_count().collect()

date,lon,lat,flooded,jrc_perm_water,precip_1d,precip_3d,NDVI,NDWI,landcover,elevation,slope,aspect,upstream_area,TWI,target
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 6. Statistik Deskriptif
Ringkasan numerik (rata-rata, simpangan baku, kuartil, min, max) untuk mendeteksi skala nilai
dan calon outlier. Data berukuran sekitar 188 MB sehingga aman di-`collect()` ke memori.

In [8]:
lf.select(pl.exclude("date")).collect().describe()

statistic,lon,lat,flooded,jrc_perm_water,precip_1d,precip_3d,NDVI,NDWI,landcover,elevation,slope,aspect,upstream_area,TWI,target
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6,1.025801e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",120.177808,-3.961987,0.112058,0.093222,11.898757,42.717388,5995.309842,0.24714,11.128318,125.314918,1.062715,155.005117,16.332991,2.843527,0.045189
"""std""",0.52951,1.018204,0.315438,0.290743,14.388524,31.635749,2020.938202,0.116986,3.762334,274.432429,1.597632,106.337275,149.262247,2.500571,0.207719
"""min""",119.353537,-6.495942,0.0,0.0,0.0,0.0,-2000.0,-0.385959,1.0,0.0,0.0,0.0,0.008497,0.000023,0.0
"""25%""",119.791466,-4.766685,0.0,0.0,0.0,20.839432,4912.0,0.183155,9.0,11.0,0.259303,63.489535,0.023397,0.933785,0.0
"""50%""",120.153038,-4.000872,0.0,0.0,7.932389,39.32737,6417.0,0.255126,12.0,25.0,0.492959,153.378298,0.08753,2.119129,0.0
"""75%""",120.357405,-2.853274,0.0,0.0,20.105299,60.266768,7552.5,0.326531,14.0,90.0,1.187444,243.542309,0.58918,4.174976,0.0
"""max""",121.790218,-1.894322,1.0,1.0,107.288399,220.21907,10000.0,0.598126,17.0,2694.0,26.720255,359.348113,5128.527832,14.996599,1.0


## 7. Profil Temporal
Memeriksa rentang tanggal dan jumlah tanggal unik. Ini menentukan apakah data cocok untuk
analisis time series atau lebih cocok sebagai snapshot antar kejadian.

In [9]:
# cek rentang tanggal awal, tanggal akhir, dan jumlah tanggal unik
lf.select(
    pl.col("date").min().alias("tanggal_awal"),
    pl.col("date").max().alias("tanggal_akhir"),
    pl.col("date").n_unique().alias("tanggal_unik"),
).collect()

tanggal_awal,tanggal_akhir,tanggal_unik
date,date,u32
2003-12-10,2015-02-05,7


In [10]:
# menghitung statistik kejadian banjir dan curah hujan per tanggal
(lf.group_by("date")
   .agg(
       pl.len().alias("n_titik"),
       pl.col("target").mean().alias("rasio_banjir"),
       pl.col("precip_1d").mean().alias("rata_hujan_1d"),
       pl.col("precip_3d").mean().alias("rata_hujan_3d"),
   )
   .sort("date")
   .collect())

date,n_titik,rasio_banjir,rata_hujan_1d,rata_hujan_3d
date,u32,f64,f64,f64
2003-12-10,146543,0.118279,11.475199,62.850881
2004-05-07,146543,0.055888,16.438275,30.301756
2006-06-19,146543,0.011724,7.718537,43.882215
2007-06-16,146543,0.04845,1.936977,39.478446
2007-07-25,146543,0.036133,6.88409,25.781169
2013-01-03,146543,0.019141,17.762995,57.099094
2015-02-05,146543,0.026709,21.075226,39.628155


## 8. Profil Spasial
Memeriksa cakupan koordinat dan jumlah titik grid unik.

In [11]:
lf.select(
    pl.col("lon").min().alias("lon_min"), pl.col("lon").max().alias("lon_max"),
    pl.col("lat").min().alias("lat_min"), pl.col("lat").max().alias("lat_max"),
    pl.struct(["lon", "lat"]).n_unique().alias("titik_grid_unik"),
).collect()

lon_min,lon_max,lat_min,lat_max,titik_grid_unik
f64,f64,f64,f64,u32
119.353537,121.790218,-6.495942,-1.894322,146543


## 9. Distribusi Label dan Definisi `target`
Melihat keseimbangan kelas serta hubungan antara `flooded`, `jrc_perm_water`, dan `target`.

In [12]:
(lf.group_by("target", "flooded", "jrc_perm_water")
   .agg(pl.len().alias("n"))
   .sort("target", "flooded")
   .collect())

target,flooded,jrc_perm_water,n
i64,f64,f64,u32
0,0.0,0.0,883819
0,0.0,1.0,27033
0,1.0,1.0,68594
1,1.0,0.0,46355


In [13]:
# Verifikasi: apakah target = flooded AND bukan air permanen?
lf.select(
    ((pl.col("target") == 1) == ((pl.col("flooded") == 1) & (pl.col("jrc_perm_water") == 0)))
    .all().alias("aturan_target_konsisten")
).collect()

aturan_target_konsisten
bool
true


In [14]:
rasio = lf.select(pl.col("target").mean()).collect().item()
print(f"Proporsi kelas banjir (target=1): {rasio:.2%}")

Proporsi kelas banjir (target=1): 4.52%


## 10. Kolom Kategorikal: `landcover`
Jumlah data dan rasio banjir per kelas tutupan lahan.

In [16]:
(lf.group_by("landcover")
   .agg(pl.len().alias("n"), pl.col("target").mean().alias("rasio_banjir"))
   .sort("landcover")
   .collect())

landcover,n,rasio_banjir
f64,u32,f64
1.0,169,0.094675
2.0,83259,0.002702
4.0,89,0.0
5.0,298,0.0
6.0,32,0.0
…,…,…
12.0,275882,0.0294
13.0,16419,0.007735
14.0,193281,0.027157


## Ringkasan Temuan Profiling

Berdasarkan hasil analisis dan profiling data menggunakan Polars, adapun poin-poin utama yang ditemukan sebagai berikut:

- Dataset memiliki 1.025.801 baris dan 16 kolom (sekitar 188 MB), memenuhi syarat lebih dari 1 juta baris.
- Pengecekan missing value menunjukan dataset tidak ada yang kosong.
- Pada pariode waktu dari 10 desember 2003 s.d 05 februari 2015 hanya ada 7 tanggal unik, dengan grid yang sama (146.543 titik) pada tiap tanggal. Berdasakan hasil tersebut data lebih tepat untuk dianalisis secara spasial dan sebagai perbandingan antar kejadian, bukan time series harian.
- Target class atau label, menunjukan `target = 1` hanya sekitar 4,5%, sehingga kelas sangat tidak seimbang. `target` konsisten dengan aturan `flooded = 1` dan `jrc_perm_water = 0`.